In [1]:
import os
import math
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
EPS = 1e-8

In [4]:
class RCModel(nn.Module):
    def __init__(self, init_parameters=None, device="cpu", dt_seconds=1800.0):
        super().__init__()
        self.device = device
        self.dt = float(dt_seconds)
        
        if init_parameters is None:
            init = {
                'Ri': 2.0, 'Re': 2.0, 'Ci': 1e5, 'Ce': 1e5,  # Ci/Ce in J/K scale
                'Ai': 10.0, 'Ae': 10.0
            }
        else:
            init = init_parameters
         
        self._raw_Ri = nn.Parameter(torch.tensor(float(init['Ri']), device=device))
        self._raw_Re = nn.Parameter(torch.tensor(float(init['Re']), device=device))
        self._raw_Ci = nn.Parameter(torch.tensor(float(init['Ci']), device=device))
        self._raw_Ce = nn.Parameter(torch.tensor(float(init['Ce']), device=device))
        self._raw_Ai = nn.Parameter(torch.tensor(float(init['Ai']), device=device))
        self._raw_Ae = nn.Parameter(torch.tensor(float(init['Ae']), device=device))
        
           
    def positive(self, raw):
        # softplus to ensure positivity (and keep gradients smooth)
        return torch.nn.functional.softplus(raw) + 1e-8

    def forward(self, Tin, Te, Tout, Phi_h, Phi_s):
        """
        Inputs: all tensors shape (batch,)
        Returns: Tin_next, Te_next (both tensors shape (batch,))
        Equations follow eq. (4) in the paper; using forward Euler with dt.
        """
        Ri = self.positive(self._raw_Ri)
        Re = self.positive(self._raw_Re)
        Ci = self.positive(self._raw_Ci)
        Ce = self.positive(self._raw_Ce)
        Ai = self.positive(self._raw_Ai)
        Ae = self.positive(self._raw_Ae)

        # coefficients:
        # dTin/dt = -1/(Ri*Ci) * Tin + 1/(Ri*Ci) * Te + 1/Ci * Phi_h + (Ai/Ci) * Phi_s
        # dTe/dt  = 1/(Ri*Ci) * Tin - (1/(Ri*Ci) + 1/(Re*Ce)) * Te + 1/(Re*Ce) * Tout + (Ae/Ce) * Phi_s
        inv_RiCi = 1.0 / (Ri * Ci)
        inv_ReCe = 1.0 / (Re * Ce)
        inv_Ci = 1.0 / Ci
        inv_Ce = 1.0 / Ce

        dTin_dt = - inv_RiCi * Tin + inv_RiCi * Te + inv_Ci * Phi_h + (Ai * inv_Ci) * Phi_s
        dTe_dt  = + inv_RiCi * Tin - (inv_RiCi + inv_ReCe) * Te + inv_ReCe * Tout + (Ae * inv_Ce) * Phi_s

        Tin_next = Tin + dTin_dt * self.dt
        Te_next = Te + dTe_dt * self.dt

        return Tin_next, Te_next

    def get_param_dict(self):
        with torch.no_grad():
            return {
                'Ri': float(self.positive(self._raw_Ri).cpu().numpy()),
                'Re': float(self.positive(self._raw_Re).cpu().numpy()),
                'Ci': float(self.positive(self._raw_Ci).cpu().numpy()),
                'Ce': float(self.positive(self._raw_Ce).cpu().numpy()),
                'Ai': float(self.positive(self._raw_Ai).cpu().numpy()),
                'Ae': float(self.positive(self._raw_Ae).cpu().numpy()),
            }

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim = 16, hidden_dims=(128,128), activation=nn.ReLU):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(activation())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(-1)

In [6]:
class PINN(nn.Module):
    def __init__(self, input_dim=16, nn_hidden=(128,128), dt_seconds=1800.0, device='cpu'):
        super().__init__()
        self.device = device
        self.nn_model = NeuralNetwork(input_dim=input_dim, hidden_dims=nn_hidden).to(device)
        self.rc_model = RCModel(dt_seconds=dt_seconds, device=device).to(device)

        # initialize near 1.0
        self._raw_lambda = nn.Parameter(torch.tensor(0.69, device=device))  # softplus ~0.9 -> lambda ~ ~?
        # helper smoothing for adaptive lambda moving average
        self.register_buffer('_lambda_ma', torch.tensor(1.0, device=device))

    def lambda_phys(self):
        # positive lambda
        return torch.nn.functional.softplus(self._raw_lambda) + 1e-8

    def forward(self, x, Tin, Te, Tout, Phi_h, Phi_s):
        """
        x: (batch, input_dim) - state + action
        Tin, Te, Tout, Phi_h, Phi_s: (batch,) tensors
        returns Tin_pred_NN, Tin_pred_RC (both shape batch,)
        """
        Tin_pred_NN = self.nn_model(x)  # shape (batch,)
        Tin_pred_RC, Te_pred_RC = self.rc_model(Tin, Te, Tout, Phi_h, Phi_s)
        return Tin_pred_NN, Tin_pred_RC

    def predict_next_state(self, x, Tin, Te, Tout, Phi_h, Phi_s):
        """
        Convenience wrapper for planning:
        Inputs can be single sample (1D) or batch (2D)
        Returns: Tin_nn (numpy), Tin_rc (numpy)
        """
        self.eval()
        with torch.no_grad():
            # ensure tensors
            if not torch.is_tensor(x): x = torch.tensor(x, dtype=torch.float32, device=self.device)
            if x.dim() == 1: x = x.unsqueeze(0)
            # inputs:
            Tin = _ensure_tensor(Tin, self.device)
            Te  = _ensure_tensor(Te, self.device)
            Tout = _ensure_tensor(Tout, self.device)
            Phi_h = _ensure_tensor(Phi_h, self.device)
            Phi_s = _ensure_tensor(Phi_s, self.device)
            tin_nn, tin_rc = self.forward(x, Tin, Te, Tout, Phi_h, Phi_s)
            return tin_nn.detach().cpu().numpy(), tin_rc.detach().cpu().numpy()

    def get_lambda(self):
        return float(self.lambda_phys().detach().cpu().numpy())

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)

    def load(self, path, map_location=None):
        map_location = map_location or self.device
        self.load_state_dict(torch.load(path, map_location=map_location))

In [7]:
def _ensure_tensor(v, device):
    if not torch.is_tensor(v):
        v = torch.tensor(v, dtype=torch.float32, device=device)
    else:
        v = v.to(device).float()
    if v.dim() == 0:
        v = v.unsqueeze(0)
    return v

In [8]:
def pinn_losses_and_terms(pinn: PINN, batch):
    """
    Compute terms:
    - loss_data = MSE(Tin_nn, Tin_actual)
    - loss_phys = MSE(Tin_nn, Tin_rc)
    - return both terms and Tin_nn for debug
    batch is a dict with keys:
      x: (batch, input_dim)
      Tin: (batch,)
      Te: (batch,)
      Tout: (batch,)
      Phi_h: (batch,)
      Phi_s: (batch,)
      Tin_actual: (batch,)  (target next Tin from env)
    """
    x = batch['x'].to(pinn.device).float()
    Tin = batch['Tin'].to(pinn.device).float()
    Te  = batch['Te'].to(pinn.device).float()
    Tout = batch['Tout'].to(pinn.device).float()
    Phi_h = batch['Phi_h'].to(pinn.device).float()
    Phi_s = batch['Phi_s'].to(pinn.device).float()
    Tin_actual = batch['Tin_actual'].to(pinn.device).float()

    Tin_nn, Tin_rc = pinn(x, Tin, Te, Tout, Phi_h, Phi_s)

    loss_data = torch.mean( (Tin_nn - Tin_actual) ** 2 )
    loss_phys = torch.mean( (Tin_nn - Tin_rc) ** 2 )
    return loss_data, loss_phys, Tin_nn, Tin_rc

In [9]:
def compute_grad_norms(pinn: PINN, loss_data, loss_phys):
    """
    Compute gradient norms w.r.t. NN parameters (we consider NN params only for balancing).
    Return norm_data, norm_phys (scalars).
    We compute gradients via autograd.grad (no .backward() here).
    """
    # Only consider parameters of the NN (not RC params or lambda param)
    nn_params = [p for p in pinn.nn_model.parameters() if p.requires_grad]
    if len(nn_params) == 0:
        return 0.0, 0.0

    grads_data = torch.autograd.grad(loss_data, nn_params, retain_graph=True, create_graph=True)
    grads_phys = torch.autograd.grad(loss_phys, nn_params, retain_graph=True, create_graph=True)

    # flatten and compute L2 norms
    def flatten_norm(grads):
        sq = 0.0
        for g in grads:
            if g is None:
                continue
            sq = sq + torch.sum(g.detach()**2)
        return torch.sqrt(sq + EPS)

    norm_data = flatten_norm(grads_data)
    norm_phys = flatten_norm(grads_phys)
    return norm_data, norm_phys

In [10]:
def update_adaptive_lambda(pinn: PINN, norm_data, norm_phys, alpha=0.9):
    """
    Adaptive lambda update rule based on gradient magnitudes:
    lambda_new := (norm_data / (norm_phys + eps))
    Then we update the raw param of lambda with EMA to avoid instability.

    alpha: moving average factor for _lambda_ma update in the model (0..1).
    Returns the updated lambda value (float).
    """
    # avoid division by zero
    ratio = (norm_data.detach() / (norm_phys.detach() + EPS)).clamp(1e-6, 1e6)
    # we will set target lambda to ratio
    target_lambda = ratio

    # store via model's internal moving average buffer for stability
    # get current ma:
    old_ma = pinn._lambda_ma.clone()
    new_ma = alpha * old_ma + (1.0 - alpha) * target_lambda
    pinn._lambda_ma.copy_(new_ma.detach())

    # now set raw lambda such that softplus(raw) ≈ new_ma
    # softplus(raw) ≈ new_ma  => raw ≈ log(exp(new_ma)-1)
    # but ensure numeric stability: if new_ma is small, use inverse softplus carefully
    new_val = new_ma.clamp(min=1e-6).to(pinn.device)
    # inverse softplus:
    raw_target = torch.log(torch.exp(new_val) - 1.0 + 1e-8)
    with torch.no_grad():
        pinn._raw_lambda.copy_(raw_target)

    return float(pinn.lambda_phys().detach().cpu().item())

In [11]:
def train_pinn_on_batch(pinn: PINN, optimizer, batch, adapt_lambda=True, lambda_alpha=0.9, clip_grad=None):
    """
    Single training step on provided batch (already tensors on device).
    batch keys: same as pinn_losses_and_terms expects.
    Returns dict with losses and lambda.
    """
    pinn.train()
    optimizer.zero_grad()

    # compute terms
    loss_data, loss_phys, Tin_nn, Tin_rc = pinn_losses_and_terms(pinn, batch)

    # optionally compute adaptive lambda using gradient norms
    if adapt_lambda:
        # compute gradient norms (w.r.t. NN params)
        norm_data, norm_phys = compute_grad_norms(pinn, loss_data, loss_phys)
        # if either norm is zero (degenerate) fallback to 1.0
        try:
            updated_lambda = update_adaptive_lambda(pinn, norm_data, norm_phys, alpha=lambda_alpha)
        except Exception:
            # safe fallback
            updated_lambda = pinn.get_lambda()
    else:
        updated_lambda = pinn.get_lambda()

    # total loss
    total_loss = loss_data + pinn.lambda_phys() * loss_phys

    # backprop & step
    total_loss.backward()
    if clip_grad is not None:
        torch.nn.utils.clip_grad_norm_(pinn.parameters(), clip_grad)
    optimizer.step()

    return {
        'loss_total': float(total_loss.detach().cpu().item()),
        'loss_data': float(loss_data.detach().cpu().item()),
        'loss_phys': float(loss_phys.detach().cpu().item()),
        'lambda': float(pinn.lambda_phys().detach().cpu().item())
    }

In [12]:
def create_pinn_and_optimizer(input_dim=16, hidden=(128,128), dt_seconds=1800.0, lr=1e-3, device='cpu'):
    device = torch.device(device)
    pinn = PINN(input_dim=input_dim, nn_hidden=hidden, dt_seconds=dt_seconds, device=device).to(device)
    optimizer = optim.Adam([
            {'params': pinn.nn_model.parameters(), 'lr': lr},
            {'params': pinn.rc_model.parameters(), 'lr': lr * 0.1},  # RC params may need smaller LR
            {'params': [pinn._raw_lambda], 'lr': lr * 0.1}
        ], lr=lr)
    return pinn, optimizer

In [13]:
if __name__ == "__main__":
    # quick sanity check (toy data)
    device = 'cpu'
    input_dim = 16
    pinn, opt = create_pinn_and_optimizer(input_dim=input_dim, hidden=(64,64), dt_seconds=1800.0, lr=1e-3, device=device)

    # make a small random batch - in practice take transitions from Dr:
    batch_size = 8
    batch = {
        'x': torch.randn(batch_size, input_dim, device=device),
        'Tin': torch.randn(batch_size, device=device) + 293.15,  # Kelvin scale example
        'Te': torch.randn(batch_size, device=device) + 293.15,
        'Tout': torch.randn(batch_size, device=device) + 283.15,
        'Phi_h': torch.randn(batch_size, device=device).abs(),   # heat power >=0
        'Phi_s': torch.randn(batch_size, device=device).abs(),   # solar irradiance >=0
        'Tin_actual': torch.randn(batch_size, device=device) + 293.15
    }

    stats = train_pinn_on_batch(pinn, opt, batch, adapt_lambda=True)
    print("Train stats:", stats)
    print("RC params:", pinn.rc_model.get_param_dict())
    print("Lambda:", pinn.get_lambda())

Train stats: {'loss_total': 172164.484375, 'loss_data': 85947.6875, 'loss_phys': 86231.0, 'lambda': 0.9997720122337341}
RC params: {'Ri': 2.1268398761749268, 'Re': 2.1269280910491943, 'Ci': 100000.0, 'Ce': 100000.0, 'Ai': 9.999945640563965, 'Ae': 10.000045776367188}
Lambda: 0.9997720122337341
